In [1]:
#!/usr/bin/env python3
"""
CD276 (B7-H3) GBM Analysis — COMPLETE Single-Cell Script
=========================================================
Generates ALL figures and ALL statistics for Scientific Reports submission.
Run on Kaggle in a SINGLE cell. ~16GB RAM sufficient.

Outputs:
  Figure1_Prognostic_Validation.png   — KM survival (TCGA + CPTAC + RNA-protein)
  Figure2_CD276_vs_PDL1.png           — Head-to-head CD276 vs PD-L1
  Figure3_SingleCell.png              — 20 cell types with Neoplastic/Non-neoplastic
  Figure4_EffluxPump_KeyFinding.png   — Efflux pumps + volcano + CD276/MGMT
  SuppFig_ForestPlot.png              — Forest plot across all cohorts/models
  SuppFig_PathwayEnrichment.png       — Gene set enrichment + ABC transporter detail
"""

import subprocess
subprocess.check_call(['pip', 'install', 'lifelines', '-q'])

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
import os, h5py

# ============================================================
# CONFIG
# ============================================================
TCGA_PAN  = "/kaggle/input/gbmtcgapan/gbm_tcga_pan_can_atlas_2018"
TCGA_FIRE = "/kaggle/input/gbmtcgafirehose/gbm_tcga"
TCGA_2013 = "/kaggle/input/gbmtcga2013/gbm_tcga_pub2013"
CPTAC     = "/kaggle/input/gbmcptac2021/gbm_cptac_2021"
SC_CORE   = "/kaggle/input/gbmcore/GBMCore.h5ad"
OUTDIR    = "/kaggle/working"
os.makedirs(OUTDIR, exist_ok=True)

plt.rcParams.update({'font.family':'serif','font.size':10,'axes.titlesize':11,
    'axes.labelsize':10,'xtick.labelsize':9,'ytick.labelsize':9,
    'figure.dpi':300,'savefig.dpi':300,'savefig.bbox':'tight'})

# ============================================================
# HELPER FUNCTIONS
# ============================================================
def load_cbioportal(path):
    expr = None
    for f in ["data_mrna_seq_v2_rsem.txt","data_mrna_seq_fpkm.txt","data_mrna_seq_rpkm.txt"]:
        fp = os.path.join(path, f)
        if os.path.exists(fp):
            expr = pd.read_csv(fp, sep='\t', index_col=0)
            print(f"  Expr: {f} shape={expr.shape}")
            break
    if expr is not None:
        if 'Hugo_Symbol' in expr.columns: expr = expr.set_index('Hugo_Symbol')
        if 'Entrez_Gene_Id' in expr.columns: expr = expr.drop('Entrez_Gene_Id', axis=1, errors='ignore')
        elif len(expr.columns) > 0 and expr.columns[0] in ['Entrez_Gene_Id','Entrez_Gene_Id.1']:
            expr = expr.drop(expr.columns[0], axis=1, errors='ignore')

    clin = None
    for f in ["data_clinical_patient.txt","data_clinical.txt"]:
        fp = os.path.join(path, f)
        if os.path.exists(fp):
            with open(fp) as fh: lines = fh.readlines()
            hi = next(i for i,l in enumerate(lines) if not l.startswith('#'))
            clin = pd.read_csv(fp, sep='\t', skiprows=range(0,hi), header=[0])
            for _ in range(2):
                if clin.shape[0]>2 and clin.iloc[0,0] in ['STRING','NUMBER','Sample Identifier','Patient Identifier']:
                    clin = clin.iloc[1:]
            clin = clin.reset_index(drop=True)
            break

    csamp = None
    fp = os.path.join(path, "data_clinical_sample.txt")
    if os.path.exists(fp):
        with open(fp) as fh: lines = fh.readlines()
        hi = next(i for i,l in enumerate(lines) if not l.startswith('#'))
        csamp = pd.read_csv(fp, sep='\t', skiprows=range(0,hi), header=[0])
        for _ in range(2):
            if csamp.shape[0]>2 and csamp.iloc[0,0] in ['STRING','NUMBER','Sample Identifier']:
                csamp = csamp.iloc[1:]
        csamp = csamp.reset_index(drop=True)

    prot = None
    for f in ["data_protein_quantification.txt","data_rppa.txt"]:
        fp = os.path.join(path, f)
        if os.path.exists(fp):
            prot = pd.read_csv(fp, sep='\t', index_col=0)
            if 'Hugo_Symbol' in prot.columns: prot = prot.set_index('Hugo_Symbol')
            if 'Entrez_Gene_Id' in prot.columns: prot = prot.drop('Entrez_Gene_Id', axis=1, errors='ignore')
            break
    return expr, clin, csamp, prot


def get_survival_data(clin, csamp=None):
    df = clin.copy()
    if csamp is not None:
        for col in ['PATIENT_ID','Patient Identifier','SAMPLE_ID']:
            if col in df.columns and col in csamp.columns:
                df = df.merge(csamp, on=col, how='left', suffixes=('','_s'))
                break
    pid_col = os_m = os_s = None
    for c in df.columns:
        cu = c.upper().replace(' ','_').replace('-','_')
        if pid_col is None and cu == 'PATIENT_ID': pid_col = c
        if os_m is None and 'OS_MONTHS' in cu: os_m = c
        if os_s is None and 'OS_STATUS' in cu: os_s = c
        if os_s is None and 'VITAL_STATUS' in cu: os_s = c
    if pid_col is None:
        for c in df.columns:
            if 'PATIENT' in c.upper() and 'ID' in c.upper(): pid_col=c; break
    r = pd.DataFrame()
    r['patient_id'] = df[pid_col].values if pid_col else df.index.astype(str).values
    if os_m:
        r['os_months'] = pd.to_numeric(df[os_m], errors='coerce')
    else:
        dc = cc = None
        for c in df.columns:
            cu=c.upper()
            if 'DEATH' in cu and 'DAY' in cu: dc=c
            if 'LAST_CONTACT' in cu and 'DAY' in cu: cc=c
            if 'FOLLOW_UP' in cu and 'DAY' in cu and cc is None: cc=c
        if dc or cc:
            dd = pd.to_numeric(df[dc],errors='coerce') if dc else pd.Series(np.nan,index=df.index)
            cd = pd.to_numeric(df[cc],errors='coerce') if cc else pd.Series(np.nan,index=df.index)
            r['os_months'] = dd.fillna(cd)/30.44
            if os_s is None: r['os_status'] = (~dd.isna()).astype(int)
    if os_s:
        r['os_status'] = df[os_s].astype(str).apply(
            lambda x:1 if any(k in str(x).upper() for k in ['DECEASED','DEAD','DEATH','EVENT']) or x.strip()=='1' or x.strip().startswith('1:') else 0)
    return r


def extract_gene(expr, gene):
    if gene in expr.index:
        v = expr.loc[gene]
        return (v.iloc[0] if isinstance(v,pd.DataFrame) else v).astype(float)
    pm = [g for g in expr.index if str(g).split('|')[0]==gene]
    if pm:
        v = expr.loc[pm[0]]
        return (v.iloc[0] if isinstance(v,pd.DataFrame) else v).astype(float)
    return None


def km_analysis(expr, surv_df, gene, ax, title, color_hi='#D32F2F', color_lo='#1976D2'):
    ge = extract_gene(expr, gene)
    if ge is None: return None
    common = set(ge.index) & set(surv_df['patient_id'].values)
    if not common:
        gs = {s[:12]:s for s in ge.index}
        ss = {s[:12]:s for s in surv_df['patient_id'].values}
        cs = set(gs.keys())&set(ss.keys())
        if cs:
            ge = pd.Series({ss[k]:float(ge[gs[k]]) for k in cs if not pd.isna(ge[gs[k]])})
            common = set(ge.index)&set(surv_df['patient_id'].values)
    if len(common)<20: return None
    sv = surv_df[surv_df['patient_id'].isin(common)].copy().set_index('patient_id')
    sv[gene] = ge[sv.index]; sv = sv.dropna()
    med = sv[gene].median()
    sv['grp'] = (sv[gene]>=med).map({True:'High',False:'Low'})
    hi,lo = sv[sv['grp']=='High'], sv[sv['grp']=='Low']
    lr = logrank_test(hi['os_months'],lo['os_months'],hi['os_status'],lo['os_status'])
    cox_df = sv[['os_months','os_status',gene]].copy()
    cox_df[gene] = (cox_df[gene]-cox_df[gene].mean())/cox_df[gene].std()
    try:
        cph=CoxPHFitter(); cph.fit(cox_df,duration_col='os_months',event_col='os_status')
        hr=np.exp(cph.params_[gene]); ci=np.exp(cph.confidence_intervals_.values[0])
        c_idx=cph.concordance_index_
    except: hr=np.nan; ci=[np.nan,np.nan]; c_idx=np.nan
    kmf_hi=KaplanMeierFitter(); kmf_lo=KaplanMeierFitter()
    kmf_hi.fit(hi['os_months'],hi['os_status'],label=f'{gene}-High (n={len(hi)})')
    kmf_lo.fit(lo['os_months'],lo['os_status'],label=f'{gene}-Low (n={len(lo)})')
    kmf_hi.plot_survival_function(ax=ax,color=color_hi,ci_show=True,ci_alpha=0.15)
    kmf_lo.plot_survival_function(ax=ax,color=color_lo,ci_show=True,ci_alpha=0.15)
    ax.set_title(title,fontweight='bold'); ax.set_xlabel('Overall Survival (months)')
    ax.set_ylabel('Survival Probability'); ax.set_ylim(0,1.05)
    ax.legend(loc='upper right',fontsize=8,framealpha=0.9)
    mhi,mlo = kmf_hi.median_survival_time_, kmf_lo.median_survival_time_
    ax.text(0.05,0.05,f"Log-rank p={lr.p_value:.3f}\n\u0394={mlo-mhi:.1f} mo",
        transform=ax.transAxes,fontsize=8,va='bottom',bbox=dict(boxstyle='round,pad=0.3',facecolor='wheat',alpha=0.8))
    r = {'n':len(sv),'n_hi':len(hi),'n_lo':len(lo),'median_hi':mhi,'median_lo':mlo,
         'logrank_p':lr.p_value,'hr':hr,'hr_ci':ci,'c_index':c_idx,
         'expression_cv':sv[gene].std()/sv[gene].mean() if sv[gene].mean()!=0 else np.nan}
    print(f"  {gene}: n={r['n']}, Hi={mhi:.1f} vs Lo={mlo:.1f}, p={lr.p_value:.4f}, HR={hr:.2f}, C={c_idx:.3f}")
    return r


def read_h5ad_categorical(f, group_path):
    """Read a potentially-categorical column from h5ad via h5py."""
    obj = f[group_path] if isinstance(group_path,str) and '/' in group_path else group_path
    if isinstance(obj, h5py.Group):
        cats = obj['categories'][:]; codes = obj['codes'][:]
        if isinstance(cats[0],bytes): cats = np.array([c.decode() for c in cats])
        return cats[codes]
    elif isinstance(obj, h5py.Dataset):
        v = obj[:]
        if len(v)>0 and isinstance(v[0],bytes): v = np.array([x.decode() for x in v])
        return v
    return None


# ============================================================
# LOAD ALL DATA
# ============================================================
print("="*70); print("LOADING ALL DATASETS"); print("="*70)

tcga = {}
for name,path in [("PanCancer",TCGA_PAN),("Firehose",TCGA_FIRE),("2013",TCGA_2013)]:
    print(f"\n--- TCGA {name} ---")
    e,c,cs,_ = load_cbioportal(path)
    if e is None: continue
    s = get_survival_data(c,cs); s = s.dropna(subset=['os_months','os_status'])
    print(f"  Survival: {len(s)} patients")
    tcga[name] = {'expr':e,'surv':s,'clin':c,'csamp':cs}

print(f"\n--- CPTAC ---")
cptac_e,cptac_c,cptac_cs,cptac_p = load_cbioportal(CPTAC)
cptac_s = get_survival_data(cptac_c,cptac_cs); cptac_s = cptac_s.dropna(subset=['os_months','os_status'])
if cptac_p is not None and any('|' in str(g) for g in cptac_p.index[:5]):
    cptac_p.index = [str(g).split('|')[0] for g in cptac_p.index]
print(f"  Survival: {len(cptac_s)} patients")

# ============================================================
# FIGURE 1: Prognostic Validation
# ============================================================
print("\n"+"="*70); print("FIGURE 1: Prognostic Validation"); print("="*70)
fig1,ax1 = plt.subplots(1,3,figsize=(14,4.5))
cd276_stats = {}

pk = next((k for k in ["PanCancer","Firehose","2013"] if k in tcga),None)
if pk:
    r = km_analysis(tcga[pk]['expr'],tcga[pk]['surv'],'CD276',ax1[0],'(a) TCGA Discovery (CD276)')
    if r: cd276_stats['tcga'] = r

if cptac_p is not None:
    r = km_analysis(cptac_p,cptac_s,'CD276',ax1[1],'(b) CPTAC Protein-Level',color_hi='#C62828',color_lo='#0D47A1')
    if r: cd276_stats['cptac_protein'] = r

# RNA-protein correlation
if cptac_e is not None and cptac_p is not None:
    rna = extract_gene(cptac_e,'CD276'); pro = extract_gene(cptac_p,'CD276')
    if rna is not None and pro is not None:
        cs2 = set(rna.index)&set(pro.index)
        if not cs2:
            rs={s[:12]:s for s in rna.index if not pd.isna(rna[s])}
            ps={s[:12]:s for s in pro.index if not pd.isna(pro[s])}
            cs2k = set(rs.keys())&set(ps.keys())
            rv = np.array([float(rna[rs[k]]) for k in cs2k])
            pv = np.array([float(pro[ps[k]]) for k in cs2k])
        else:
            rv = np.array([float(rna[s]) for s in cs2 if np.isfinite(rna[s]) and np.isfinite(pro[s])])
            pv = np.array([float(pro[s]) for s in cs2 if np.isfinite(rna[s]) and np.isfinite(pro[s])])
        m = np.isfinite(rv)&np.isfinite(pv); rv,pv = rv[m],pv[m]
        if len(rv)>10:
            sr,sp = stats.spearmanr(rv,pv); pr2,pp = stats.pearsonr(rv,pv)
            ax1[2].scatter(rv,pv,alpha=0.6,s=30,c='#37474F',edgecolors='white',linewidth=0.3)
            z=np.polyfit(rv,pv,1); xl=np.linspace(rv.min(),rv.max(),100)
            ax1[2].plot(xl,np.poly1d(z)(xl),'r-',linewidth=1.5,alpha=0.8)
            ax1[2].set_title('(c) RNA\u2013Protein Correlation',fontweight='bold')
            ax1[2].set_xlabel('CD276 RNA Expression'); ax1[2].set_ylabel('CD276 Protein Abundance')
            ax1[2].text(0.05,0.95,f"Spearman r={sr:.3f}\np={sp:.1e}\nn={len(rv)}",
                transform=ax1[2].transAxes,fontsize=8,va='top',bbox=dict(boxstyle='round,pad=0.3',facecolor='wheat',alpha=0.8))
            print(f"  RNA-Protein: Spearman r={sr:.3f} (p={sp:.1e}), Pearson r={pr2:.3f}, n={len(rv)}")

fig1.tight_layout(); fig1.savefig(os.path.join(OUTDIR,"Figure1_Prognostic_Validation.png")); plt.close()
print("  -> Figure 1 saved")

# ============================================================
# FIGURE 2: CD276 vs PD-L1
# ============================================================
print("\n"+"="*70); print("FIGURE 2: CD276 vs PD-L1"); print("="*70)
fig2,ax2 = plt.subplots(1,2,figsize=(10,4.5))
pdl1_stats = {}
if pk:
    km_analysis(tcga[pk]['expr'],tcga[pk]['surv'],'CD276',ax2[0],'(a) CD276 Stratification')
    r2 = km_analysis(tcga[pk]['expr'],tcga[pk]['surv'],'CD274',ax2[1],'(b) PD-L1 (CD274) Stratification',color_hi='#7B1FA2',color_lo='#00897B')
    if r2: pdl1_stats['tcga'] = r2
fig2.tight_layout(); fig2.savefig(os.path.join(OUTDIR,"Figure2_CD276_vs_PDL1.png")); plt.close()
print("  -> Figure 2 saved")

# PD-L1 in CPTAC
if cptac_p is not None:
    print("  PD-L1 protein (CPTAC):")
    rp = km_analysis(cptac_p,cptac_s,'CD274',plt.figure().add_subplot(),'tmp',color_hi='#7B1FA2',color_lo='#00897B')
    if rp: pdl1_stats['cptac_protein'] = rp
    plt.close()
if cptac_e is not None:
    print("  PD-L1 RNA (CPTAC):")
    rr = km_analysis(cptac_e,cptac_s,'CD274',plt.figure().add_subplot(),'tmp',color_hi='#7B1FA2',color_lo='#00897B')
    if rr: pdl1_stats['cptac_rna'] = rr
    plt.close()

# ============================================================
# FIGURE 3: Single-Cell (all 20 types, Neoplastic vs Non-neoplastic)
# ============================================================
print("\n"+"="*70); print("FIGURE 3: Single-Cell Analysis"); print("="*70)

sc_data = None
try:
    with h5py.File(SC_CORE,'r') as f:
        var_names = f['var']['_index'][:]
        if isinstance(var_names[0],bytes): var_names = np.array([v.decode() for v in var_names])
        gene_symbols = read_h5ad_categorical(f, f['var']['feature_name'])
        cd276_idx = np.where(gene_symbols=='CD276')[0]
        if len(cd276_idx)==0: cd276_idx = np.where(var_names=='ENSG00000103855')[0]
        cd276_idx = cd276_idx[0]
        print(f"  CD276 at index {cd276_idx}")

        # Read all annotation levels
        ann = {}
        for lv in ['annotation_level_1','annotation_level_2','annotation_level_3','annotation_level_4']:
            if lv in f['obs']: ann[lv] = read_h5ad_categorical(f, f['obs'][lv])

        ct3 = ann.get('annotation_level_3', ann.get('annotation_level_1'))
        ct1 = ann.get('annotation_level_1')
        print(f"  Level 3: {len(np.unique(ct3))} types, {len(ct3)} cells")

        # Read CD276 from sparse X
        xg = f['X']; indptr = xg['indptr'][:]
        shape = tuple(xg.attrs.get('shape', xg.attrs.get('h5sparse_shape',(len(ct3),len(var_names)))))
        if len(indptr)==shape[1]+1:  # CSC
            s,e = int(indptr[cd276_idx]), int(indptr[cd276_idx+1])
            ri,vl = xg['indices'][s:e], xg['data'][s:e]
            cd276_sc = np.zeros(shape[0],dtype=float); cd276_sc[ri] = vl
        else:  # CSR
            idx=xg['indices'][:]; dat=xg['data'][:]
            cm = idx==cd276_idx; cd276_sc = np.zeros(shape[0],dtype=float)
            if cm.any():
                fp=np.where(cm)[0]; rids=np.searchsorted(indptr[1:],fp,side='right')
                cd276_sc[rids] = dat[fp]
            del idx,dat,cm
        print(f"  CD276: mean={cd276_sc.mean():.4f}, >0={np.sum(cd276_sc>0)}/{len(cd276_sc)}")

        # Build per-type stats
        ct_stats = []
        for ct in np.unique(ct3):
            m = ct3==ct; v = cd276_sc[m]; n=int(m.sum())
            cat = 'Neoplastic' if ct1 is not None and np.mean(ct1[m]=='Neoplastic')>0.5 else 'Non-neoplastic'
            ct_stats.append({'cell_type':ct,'category':cat,'n':n,
                'mean':float(np.mean(v)),'pct':float(np.sum(v>0)/n*100) if n>0 else 0})
        sc_data = pd.DataFrame(ct_stats).sort_values('mean',ascending=True)
        print(f"\n  Cell type breakdown ({len(sc_data)} types):")
        for _,r in sc_data.sort_values('mean',ascending=False).iterrows():
            print(f"    {r['cell_type']:25s} ({r['category']:15s}): mean={r['mean']:.4f}, pct={r['pct']:.1f}%, n={r['n']:,d}")
except Exception as ex:
    import traceback; print(f"  ERROR: {ex}"); traceback.print_exc()

fig3,ax3 = plt.subplots(1,2,figsize=(14,6))
if sc_data is not None and len(sc_data)>0:
    cmap = {'Neoplastic':'#e74c3c','Non-neoplastic':'#3498db'}
    bc = [cmap.get(r['category'],'#95a5a6') for _,r in sc_data.iterrows()]
    ax3[0].barh(range(len(sc_data)),sc_data['mean'],color=bc,height=0.7)
    ax3[0].set_yticks(range(len(sc_data))); ax3[0].set_yticklabels(sc_data['cell_type'],fontsize=8)
    ax3[0].set_xlabel('Mean CD276 Expression'); ax3[0].set_title('(a) CD276 Expression by Cell Type',fontweight='bold')
    ax3[1].barh(range(len(sc_data)),sc_data['pct'],color=bc,height=0.7)
    ax3[1].set_yticks(range(len(sc_data))); ax3[1].set_yticklabels(sc_data['cell_type'],fontsize=8)
    ax3[1].set_xlabel('% Cells Expressing CD276'); ax3[1].set_title('(b) Fraction of Cells Expressing CD276',fontweight='bold')
    ax3[1].legend(handles=[mpatches.Patch(fc='#e74c3c',label='Neoplastic'),mpatches.Patch(fc='#3498db',label='Non-neoplastic')],loc='lower right')
else:
    for a in ax3: a.text(0.5,0.5,'Single-cell data\ncould not be loaded',ha='center',va='center',transform=a.transAxes)
fig3.tight_layout(); fig3.savefig(os.path.join(OUTDIR,"Figure3_SingleCell.png")); plt.close()
print("  -> Figure 3 saved")

# ============================================================
# FIGURE 4: Efflux Pump Inverse Relationship (KEY FINDING)
# ============================================================
print("\n"+"="*70); print("FIGURE 4: Efflux Pump Analysis"); print("="*70)
fig4,ax4 = plt.subplots(1,3,figsize=(15,5))

de_results = []
if pk:
    expr = tcga[pk]['expr']
    cd276 = extract_gene(expr,'CD276')
    if cd276 is not None:
        cd276 = cd276.dropna(); med_c = cd276.median()
        hi_s = cd276[cd276>=med_c].index; lo_s = cd276[cd276<med_c].index

        for gene in ['CHI3L1','VEGFA','VIM','CD163','HIF1A','ABCG2','ABCB1','SOX2','PROM1']:
            g = extract_gene(expr,gene)
            if g is None: continue
            hv = g.reindex(hi_s).dropna().astype(float); lv = g.reindex(lo_s).dropna().astype(float)
            if len(hv)<10 or len(lv)<10: continue
            _,pv = stats.mannwhitneyu(hv,lv,alternative='two-sided')
            fc = hv.mean()/lv.mean() if lv.mean()>0 else np.nan
            l2fc = np.log2(fc) if fc>0 else 0
            de_results.append({'gene':gene,'fc':fc,'log2fc':l2fc,'pval':pv,
                'nlp':-np.log10(pv) if pv>0 else 30,'hi':hv.mean(),'lo':lv.mean()})
            print(f"  {gene:8s}: FC={fc:.2f}, p={pv:.4f}, Hi={hv.mean():.2f}, Lo={lv.mean():.2f}")

        de_df = pd.DataFrame(de_results)

        # Panel A: Efflux box plots
        pos,dp,lbl,col = 0,[],[],[]
        for gene in ['ABCG2','ABCB1']:
            g = extract_gene(expr,gene)
            if g is None: continue
            hv = g.reindex(hi_s).dropna().astype(float).values
            lv = g.reindex(lo_s).dropna().astype(float).values
            dp.extend([lv,hv]); lbl.extend([f'{gene}\nCD276-Low',f'{gene}\nCD276-High'])
            col.extend(['#1976D2','#D32F2F']); pos +=3
        bp = ax4[0].boxplot(dp,positions=[0,1,3,4],widths=0.7,patch_artist=True,showfliers=False,
            medianprops=dict(color='black',linewidth=1.5))
        for patch,c in zip(bp['boxes'],col): patch.set_facecolor(c); patch.set_alpha(0.7)
        ax4[0].set_xticks([0,1,3,4]); ax4[0].set_xticklabels(lbl,fontsize=8)
        ax4[0].set_ylabel('Expression (RSEM)'); ax4[0].set_title('(a) Drug Efflux Transporters',fontweight='bold')
        for i,gene in enumerate(['ABCG2','ABCB1']):
            row = de_df[de_df['gene']==gene]
            if len(row):
                pv2=row['pval'].values[0]; fc2=row['fc'].values[0]
                ym = max(dp[i*2].max(),dp[i*2+1].max())*1.1; xm = i*3+0.5
                ax4[0].plot([i*3,i*3,i*3+1,i*3+1],[ym,ym*1.02,ym*1.02,ym],'k-',linewidth=0.8)
                ps2 = f"p={pv2:.4f}" if pv2>=0.0001 else f"p={pv2:.1e}"
                ax4[0].text(xm,ym*1.03,f"{ps2}\nFC = {fc2:.2f}",ha='center',va='bottom',fontsize=7)

        # Panel B: Volcano
        for _,row in de_df.iterrows():
            c = '#D32F2F' if row['log2fc']>0 else '#1976D2'
            ax4[1].scatter(row['log2fc'],row['nlp'],c=c,s=60,zorder=5)
            ax4[1].annotate(row['gene'],(row['log2fc'],row['nlp']),fontsize=7,ha='center',va='bottom',xytext=(0,5),textcoords='offset points')
        ax4[1].axvline(0,color='grey',linewidth=0.5,linestyle='--')
        ax4[1].set_xlabel('log\u2082(Fold Change)\nCD276-High vs CD276-Low')
        ax4[1].set_ylabel('-log\u2081\u2080(p-value)'); ax4[1].set_title('(b) Differential Expression',fontweight='bold')

        # Panel C: Combined CD276/MGMT stratification
        surv = tcga[pk]['surv']
        mgmt = extract_gene(expr,'MGMT')
        if mgmt is not None:
            gs = {s[:12]:s for s in cd276.index}; ms = {s[:12]:s for s in mgmt.index}
            ss2 = {s[:12]:s for s in surv['patient_id'].values}
            com = set(gs.keys())&set(ms.keys())&set(ss2.keys())
            if len(com)>40:
                comb = pd.DataFrame({'patient_id':[ss2[k] for k in com],
                    'cd276':[float(cd276[gs[k]]) for k in com],'mgmt':[float(mgmt[ms[k]]) for k in com]})
                comb = comb.merge(surv,on='patient_id')
                comb = comb.dropna(subset=['os_months','os_status'])
                c_med = comb['cd276'].median(); m_med = comb['mgmt'].median()
                comb['grp'] = comb.apply(lambda r: f"CD276-{'Hi' if r['cd276']>=c_med else 'Lo'}/MGMT-{'Hi' if r['mgmt']>=m_med else 'Lo'}",axis=1)

                sp_r,sp_p = stats.spearmanr(comb['cd276'],comb['mgmt'])
                print(f"\n  CD276-MGMT correlation: r={sp_r:.3f}, p={sp_p:.4f}")

                grp_colors = {'CD276-Hi/MGMT-Hi':'#D32F2F','CD276-Hi/MGMT-Lo':'#FF9800',
                              'CD276-Lo/MGMT-Hi':'#1976D2','CD276-Lo/MGMT-Lo':'#4CAF50'}
                for gn,gc in grp_colors.items():
                    sub = comb[comb['grp']==gn]
                    if len(sub)>=3:
                        kmf=KaplanMeierFitter(); kmf.fit(sub['os_months'],sub['os_status'],label=f"{gn} (n={len(sub)})")
                        kmf.plot_survival_function(ax=ax4[2],color=gc,ci_show=False)
                        print(f"  {gn}: n={len(sub)}, median OS={kmf.median_survival_time_:.1f} mo")
                # Test worst vs best
                worst = comb[comb['grp']=='CD276-Hi/MGMT-Hi']; best = comb[comb['grp']=='CD276-Lo/MGMT-Lo']
                if len(worst)>=5 and len(best)>=5:
                    lr_wb = logrank_test(worst['os_months'],best['os_months'],worst['os_status'],best['os_status'])
                    print(f"  Worst vs Best: p={lr_wb.p_value:.4f}")
                ax4[2].set_title('(c) Combined CD276/MGMT Stratification',fontweight='bold')
                ax4[2].set_xlabel('Overall Survival (months)'); ax4[2].set_ylabel('Survival Probability')
                ax4[2].legend(fontsize=7,loc='upper right')

fig4.tight_layout(); fig4.savefig(os.path.join(OUTDIR,"Figure4_EffluxPump_KeyFinding.png")); plt.close()
print("  -> Figure 4 saved")

# ============================================================
# MULTIVARIATE COX REGRESSION (TCGA 2013 — richest clinical annotation)
# ============================================================
print("\n"+"="*70); print("MULTIVARIATE COX REGRESSION"); print("="*70)

mv_results = {}  # store for forest plot
if "2013" in tcga:
    d = tcga["2013"]; e2=d['expr']; c2=d['clin']; cs2=d['csamp']; sv2=d['surv']
    cd = extract_gene(e2,'CD276')
    if cd is not None:
        # Build merged df
        gs = {str(s)[:12]:s for s in cd.index}
        ss = {str(s)[:12]:s for s in sv2['patient_id'].values}
        com = set(gs.keys())&set(ss.keys())
        df = pd.DataFrame({'pid':[ss[k] for k in com],'CD276':[float(cd[gs[k]]) for k in com]})
        df = df.merge(sv2.rename(columns={'patient_id':'pid'}),on='pid')

        # Merge clinical covariates
        c2m = c2.copy()
        if cs2 is not None:
            for col in ['PATIENT_ID']:
                if col in c2m.columns and col in cs2.columns:
                    c2m = c2m.merge(cs2,on=col,how='left',suffixes=('','_s')); break
        c2m['_pid'] = c2m.get('PATIENT_ID',c2m.iloc[:,0]).astype(str).str[:12]
        df['_pid'] = df['pid'].astype(str).str[:12]
        df = df.merge(c2m[['_pid']+[c for c in c2m.columns if c!='_pid']],on='_pid',how='left')

        df['OS_time'] = df['os_months']; df['OS_event'] = df['os_status'].astype(int)
        df = df[df['OS_time']>0]
        df['CD276_std'] = (df['CD276']-df['CD276'].mean())/df['CD276'].std()

        # Parse covariates
        for c in df.columns:
            if c.upper()=='AGE': df['Age'] = pd.to_numeric(df[c],errors='coerce')
            if c.upper()=='SEX': df['Sex_male'] = df[c].astype(str).str.upper().str.strip().isin(['MALE','M']).astype(int)
        for c in df.columns:
            if 'MGMT' in c.upper():
                print(f"  MGMT column: {c} -> {df[c].value_counts().to_dict()}")
                df['MGMT_unmeth'] = df[c].astype(str).str.upper().str.strip().isin(['UNMETHYLATED','U','UNMETH']).astype(int)
                known = df[c].astype(str).str.upper().str.strip().isin(['METHYLATED','UNMETHYLATED','M','U'])
                df.loc[~known,'MGMT_unmeth'] = np.nan; break
        for c in df.columns:
            if 'IDH1' in c.upper():
                print(f"  IDH1 column: {c} -> {df[c].value_counts().to_dict()}")
                df['IDH1_mut'] = df[c].astype(str).str.upper().str.strip().isin(['YES','MUTATED','MUT','R132H','R132C','Y','1']).astype(int)
                bad = df[c].astype(str).str.upper().str.strip().isin(['[NOT AVAILABLE]','NAN','NA',''])
                df.loc[bad,'IDH1_mut'] = np.nan; break

        # Model 1: Univariate
        print("\n--- Univariate Cox ---")
        df_u = df[['OS_time','OS_event','CD276_std']].dropna()
        cph_u = CoxPHFitter(); cph_u.fit(df_u,duration_col='OS_time',event_col='OS_event')
        u_hr = np.exp(cph_u.params_['CD276_std']); u_ci = cph_u.confidence_intervals_.loc['CD276_std']
        u_p = cph_u.summary.loc['CD276_std','p']; u_c = cph_u.concordance_index_
        print(f"  HR={u_hr:.3f} ({np.exp(u_ci.iloc[0]):.2f}-{np.exp(u_ci.iloc[1]):.2f}), p={u_p:.4f}, C={u_c:.3f}, n={len(df_u)}")
        mv_results['2013_uni'] = {'hr':u_hr,'ci_lo':np.exp(u_ci.iloc[0]),'ci_hi':np.exp(u_ci.iloc[1]),'p':u_p,'n':len(df_u),'c':u_c}

        # Model 2: + Age + Sex
        print("\n--- Multivariate (CD276 + Age + Sex) ---")
        df_m2 = df[['OS_time','OS_event','CD276_std','Age','Sex_male']].dropna()
        cph_m2 = CoxPHFitter(); cph_m2.fit(df_m2,duration_col='OS_time',event_col='OS_event')
        m2_hr = np.exp(cph_m2.params_['CD276_std']); m2_ci = cph_m2.confidence_intervals_.loc['CD276_std']
        m2_p = cph_m2.summary.loc['CD276_std','p']; m2_c = cph_m2.concordance_index_
        print(f"  CD276 HR={m2_hr:.3f} ({np.exp(m2_ci.iloc[0]):.2f}-{np.exp(m2_ci.iloc[1]):.2f}), p={m2_p:.4f}, C={m2_c:.3f}, n={len(df_m2)}")
        cph_m2.print_summary()

        # Model 3: Full (+ MGMT + IDH1)
        print("\n--- Full Multivariate (CD276 + Age + Sex + MGMT + IDH1) ---")
        covs = ['OS_time','OS_event','CD276_std','Age','Sex_male']
        if 'MGMT_unmeth' in df.columns: covs.append('MGMT_unmeth')
        if 'IDH1_mut' in df.columns: covs.append('IDH1_mut')
        df_m3 = df[covs].dropna()
        print(f"  n={len(df_m3)} (after dropping missing)")
        cph_m3 = CoxPHFitter(); cph_m3.fit(df_m3,duration_col='OS_time',event_col='OS_event')
        m3_hr = np.exp(cph_m3.params_['CD276_std']); m3_ci = cph_m3.confidence_intervals_.loc['CD276_std']
        m3_p = cph_m3.summary.loc['CD276_std','p']; m3_c = cph_m3.concordance_index_
        print(f"  CD276 HR={m3_hr:.3f} ({np.exp(m3_ci.iloc[0]):.2f}-{np.exp(m3_ci.iloc[1]):.2f}), p={m3_p:.4f}, C={m3_c:.3f}")
        cph_m3.print_summary()
        mv_results['2013_full'] = {'hr':m3_hr,'ci_lo':np.exp(m3_ci.iloc[0]),'ci_hi':np.exp(m3_ci.iloc[1]),'p':m3_p,'n':len(df_m3),'c':m3_c}

        # PH assumption
        print("\n--- Proportional Hazards Test ---")
        try: cph_m3.check_assumptions(df_m3,p_value_threshold=0.05,show_plots=False); print("  PH assumption OK")
        except Exception as ex: print(f"  PH test: {ex}")

# Cross-pipeline univariate Cox for forest plot
for name in tcga:
    if name == "2013" and '2013_uni' in mv_results: continue
    e = tcga[name]['expr']; s = tcga[name]['surv']
    cd = extract_gene(e,'CD276')
    if cd is None: continue
    gs = {str(x)[:12]:x for x in cd.index}; ss = {str(x)[:12]:x for x in s['patient_id'].values}
    com = set(gs.keys())&set(ss.keys())
    dfx = pd.DataFrame({'pid':[ss[k] for k in com],'CD276':[float(cd[gs[k]]) for k in com]})
    dfx = dfx.merge(s.rename(columns={'patient_id':'pid'}),on='pid')
    dfx['CD276_std'] = (dfx['CD276']-dfx['CD276'].mean())/dfx['CD276'].std()
    dfx = dfx[['os_months','os_status','CD276_std']].dropna()
    dfx = dfx[dfx['os_months']>0]
    cph = CoxPHFitter(); cph.fit(dfx,duration_col='os_months',event_col='os_status')
    hr = np.exp(cph.params_['CD276_std']); ci = cph.confidence_intervals_.loc['CD276_std']
    p = cph.summary.loc['CD276_std','p']; c = cph.concordance_index_
    mv_results[f'{name}_uni'] = {'hr':hr,'ci_lo':np.exp(ci.iloc[0]),'ci_hi':np.exp(ci.iloc[1]),'p':p,'n':len(dfx),'c':c}
    print(f"  {name} univariate: HR={hr:.3f}, p={p:.4f}, C={c:.3f}, n={len(dfx)}")

# CPTAC protein Cox
if cptac_p is not None:
    cd = extract_gene(cptac_p,'CD276')
    if cd is not None:
        gs = {str(x)[:12]:x for x in cd.index}; ss = {str(x)[:12]:x for x in cptac_s['patient_id'].values}
        com = set(gs.keys())&set(ss.keys())
        dfx = pd.DataFrame({'pid':[ss[k] for k in com],'CD276':[float(cd[gs[k]]) for k in com]})
        dfx = dfx.merge(cptac_s.rename(columns={'patient_id':'pid'}),on='pid')
        dfx['CD276_std'] = (dfx['CD276']-dfx['CD276'].mean())/dfx['CD276'].std()
        dfx = dfx[['os_months','os_status','CD276_std']].dropna(); dfx = dfx[dfx['os_months']>0]
        cph = CoxPHFitter(); cph.fit(dfx,duration_col='os_months',event_col='os_status')
        hr = np.exp(cph.params_['CD276_std']); ci = cph.confidence_intervals_.loc['CD276_std']
        p = cph.summary.loc['CD276_std','p']; c = cph.concordance_index_
        mv_results['CPTAC_protein'] = {'hr':hr,'ci_lo':np.exp(ci.iloc[0]),'ci_hi':np.exp(ci.iloc[1]),'p':p,'n':len(dfx),'c':c}
        print(f"  CPTAC protein: HR={hr:.3f}, p={p:.4f}, C={c:.3f}, n={len(dfx)}")

# ============================================================
# GENE SET ENRICHMENT
# ============================================================
print("\n"+"="*70); print("GENE SET / PATHWAY ANALYSIS"); print("="*70)

gsea_results = []
if pk:
    expr = tcga[pk]['expr']; cd276 = extract_gene(expr,'CD276')
    if cd276 is not None:
        med = cd276.median(); hi_s = cd276[cd276>=med].index; lo_s = cd276[cd276<med].index
        gene_sets = {
            'ABC_TRANSPORTERS':['ABCA1','ABCA2','ABCA3','ABCB1','ABCB4','ABCB5','ABCB6','ABCB7','ABCB8','ABCB10','ABCB11',
                'ABCC1','ABCC2','ABCC3','ABCC4','ABCC5','ABCC6','ABCC8','ABCC9','ABCC10','ABCC11',
                'ABCD1','ABCD2','ABCD3','ABCE1','ABCF1','ABCF2','ABCF3','ABCG1','ABCG2','ABCG4','ABCG5','ABCG8'],
            'DRUG_EFFLUX_CORE':['ABCB1','ABCG2','ABCC1','ABCC2','ABCC3','ABCC4','ABCC5'],
            'ADC_PAYLOAD_RESISTANCE':['ABCB1','ABCG2','ABCC1','TOP1','TUBB3','TUBB','BCL2','BCL2L1','MCL1','CASP3','CASP8','CASP9'],
            'STEM_CELL_MARKERS':['SOX2','PROM1','NES','OLIG2','CD44','ALDH1A3','BMI1','EZH2','NANOG','POU5F1','NOTCH1','HES1','MYC'],
            'ANGIOGENESIS':['VEGFA','VEGFB','VEGFC','FLT1','KDR','FLT4','ANGPT1','ANGPT2','TEK','PECAM1','CDH5','ENG','HIF1A','EPAS1','NRP1','NRP2','DLL4','NOTCH1'],
            'MESENCHYMAL_GBM':['CHI3L1','VIM','SERPINE1','TGFBI','MMP2','MMP9','LGALS3','CD44','FN1','COL1A1','COL1A2','ACTA2','CD163','CD68','MRC1','MSR1'],
        }
        for gs_name,genes in gene_sets.items():
            pg = [g for g in genes if g in expr.index]
            if len(pg)<3: continue
            l2fcs = []
            for g in pg:
                hv = expr.loc[g,hi_s].astype(float).values; lv = expr.loc[g,lo_s].astype(float).values
                hm = np.mean(hv[hv>0]) if np.sum(hv>0)>0 else 1
                lm = np.mean(lv[lv>0]) if np.sum(lv>0)>0 else 1
                fc = hm/lm if lm>0 else 1; l2fcs.append(np.log2(fc) if fc>0 else 0)
            ml = np.mean(l2fcs)
            _,sp = stats.wilcoxon(l2fcs,alternative='two-sided') if len(l2fcs)>=6 else (0,1.0)
            d = "UP" if ml>0 else "DOWN"
            gsea_results.append({'set':gs_name,'n':len(pg),'mean_l2fc':ml,'p':sp,'dir':d})
            print(f"  {gs_name:30s}: {len(pg):2d} genes, mean log2FC={ml:+.3f}, p={sp:.4f} | {d} in CD276-High")

# ============================================================
# SUPPLEMENTARY FIGURE: FOREST PLOT
# ============================================================
print("\n"+"="*70); print("SUPP FIGURE: Forest Plot"); print("="*70)

forest = []
label_order = ['2013_uni','2013_full','PanCancer_uni','Firehose_uni','CPTAC_protein']
nice_labels = {'2013_uni':'TCGA 2013 (univariate)','2013_full':'TCGA 2013 (multivariate)',
    'PanCancer_uni':'TCGA PanCancer (univariate)','Firehose_uni':'TCGA Firehose (univariate)',
    'CPTAC_protein':'CPTAC protein (univariate)'}
for k in label_order:
    if k in mv_results:
        r = mv_results[k]; forest.append({**r,'label':nice_labels.get(k,k)})
        print(f"  {nice_labels.get(k,k):40s}: HR={r['hr']:.3f} ({r['ci_lo']:.2f}-{r['ci_hi']:.2f}), p={r['p']:.4f}, n={r['n']}")

fig_f,ax_f = plt.subplots(figsize=(8,3.5+0.4*len(forest)))
yp = list(range(len(forest)))[::-1]
for i,(fd,y) in enumerate(zip(forest,yp)):
    c = '#c0392b' if 'multivariate' in fd['label'].lower() else ('#2980b9' if 'CPTAC' in fd['label'] else '#2c3e50')
    ax_f.plot([fd['ci_lo'],fd['ci_hi']],[y,y],color=c,linewidth=2)
    ax_f.plot(fd['hr'],y,'o',color=c,markersize=8,zorder=5)
    ps = f"p={fd['p']:.3f}" if fd['p']>=0.001 else "p<0.001"
    ax_f.text(max(fd['ci_hi'],1.8)+0.05,y,f"HR={fd['hr']:.2f} ({fd['ci_lo']:.2f}-{fd['ci_hi']:.2f}), {ps}, n={fd['n']}",va='center',fontsize=8)
ax_f.axvline(1.0,color='grey',linestyle='--',linewidth=0.8)
ax_f.set_yticks(yp); ax_f.set_yticklabels([f['label'] for f in forest])
ax_f.set_xlabel('Hazard Ratio (per SD increase in CD276)')
ax_f.set_title('Forest Plot: CD276 Prognostic Effect Across Cohorts')
ax_f.set_xlim(0.5,2.5); ax_f.spines['top'].set_visible(False); ax_f.spines['right'].set_visible(False)
plt.tight_layout(); fig_f.savefig(os.path.join(OUTDIR,"SuppFig_ForestPlot.png")); plt.close()
print("  -> SuppFig_ForestPlot.png saved")

# ============================================================
# SUPPLEMENTARY FIGURE: PATHWAY ENRICHMENT
# ============================================================
print("\n"+"="*70); print("SUPP FIGURE: Pathway Enrichment"); print("="*70)

if gsea_results and pk:
    fig_g,axg = plt.subplots(1,2,figsize=(14,5))
    gs_df = pd.DataFrame(gsea_results).sort_values('mean_l2fc')
    bc = ['#c0392b' if x>0 else '#2980b9' for x in gs_df['mean_l2fc']]
    axg[0].barh(range(len(gs_df)),gs_df['mean_l2fc'],color=bc,height=0.6)
    axg[0].set_yticks(range(len(gs_df))); axg[0].set_yticklabels([n.replace('_',' ').title() for n in gs_df['set']])
    axg[0].axvline(0,color='grey',linewidth=0.5)
    axg[0].set_xlabel('Mean log\u2082(Fold Change)\nCD276-High vs CD276-Low')
    axg[0].set_title('(a) Gene Set Enrichment in CD276-High Tumours',fontweight='bold')
    for i,(_,row) in enumerate(gs_df.iterrows()):
        sig = '***' if row['p']<0.001 else ('**' if row['p']<0.01 else ('*' if row['p']<0.05 else ''))
        if sig:
            x=row['mean_l2fc']; off=0.02 if x>0 else -0.02
            axg[0].text(x+off,i,sig,ha='left' if x>0 else 'right',va='center',fontsize=10,fontweight='bold')

    # Panel B: ABC transporter family detail
    expr = tcga[pk]['expr']
    abc_genes = gene_sets['ABC_TRANSPORTERS']
    abc_present = [g for g in abc_genes if g in expr.index]
    gfc,gpv = [],[]
    for g in abc_present:
        hv=expr.loc[g,hi_s].astype(float).values; lv=expr.loc[g,lo_s].astype(float).values
        hm,lm=np.mean(hv),np.mean(lv); fc=hm/lm if lm>0 else 1
        l2=np.log2(fc) if fc>0 else 0
        try: _,pv=stats.mannwhitneyu(hv,lv,alternative='two-sided')
        except: pv=1.0
        gfc.append(l2); gpv.append(pv)
    si = np.argsort(gfc)
    sg = [abc_present[i] for i in si]; sf = [gfc[i] for i in si]; sp2 = [gpv[i] for i in si]
    bc2 = ['#c0392b' if x>0 else '#2980b9' for x in sf]
    axg[1].barh(range(len(sg)),sf,color=bc2,height=0.6)
    axg[1].set_yticks(range(len(sg))); axg[1].set_yticklabels(sg,fontsize=7)
    axg[1].axvline(0,color='grey',linewidth=0.5)
    axg[1].set_xlabel('log\u2082(Fold Change)\nCD276-High vs CD276-Low')
    axg[1].set_title('(b) ABC Transporter Family Detail',fontweight='bold')
    for i,pv in enumerate(sp2):
        if pv<0.05:
            sig='***' if pv<0.001 else ('**' if pv<0.01 else '*')
            x=sf[i]; off=0.02 if x>0 else -0.02
            axg[1].text(x+off,i,sig,ha='left' if x>0 else 'right',va='center',fontsize=8,fontweight='bold')
    plt.tight_layout(); fig_g.savefig(os.path.join(OUTDIR,"SuppFig_PathwayEnrichment.png")); plt.close()
    print("  -> SuppFig_PathwayEnrichment.png saved")

# ============================================================
# CROSS-PIPELINE CONSISTENCY + MGMT STRATIFICATION
# ============================================================
print("\n"+"="*70); print("CROSS-PIPELINE CONSISTENCY"); print("="*70)
for name in tcga:
    if name==pk: continue
    r = km_analysis(tcga[name]['expr'],tcga[name]['surv'],'CD276',plt.figure().add_subplot(),'tmp')
    if r: cd276_stats[f'tcga_{name}'] = r
    plt.close()

# MGMT stratification within groups
print("\n"+"="*70); print("MGMT STRATIFICATION"); print("="*70)
if pk and 'MGMT' in [extract_gene(tcga[pk]['expr'],'MGMT') is not None and 'yes'][0:0] or True:
    expr = tcga[pk]['expr']; surv = tcga[pk]['surv']
    cd276 = extract_gene(expr,'CD276'); mgmt = extract_gene(expr,'MGMT')
    if cd276 is not None and mgmt is not None:
        gs = {str(s)[:12]:s for s in cd276.index}; ms = {str(s)[:12]:s for s in mgmt.index}
        ss = {str(s)[:12]:s for s in surv['patient_id'].values}
        com = set(gs.keys())&set(ms.keys())&set(ss.keys())
        comb = pd.DataFrame({'patient_id':[ss[k] for k in com],
            'cd276':[float(cd276[gs[k]]) for k in com],'mgmt':[float(mgmt[ms[k]]) for k in com]})
        comb = comb.merge(surv,on='patient_id').dropna(subset=['os_months','os_status'])
        c_med = comb['cd276'].median(); m_med = comb['mgmt'].median()
        for label,subset in [("MGMT-High",comb[comb['mgmt']>=m_med]),("MGMT-Low",comb[comb['mgmt']<m_med])]:
            hi = subset[subset['cd276']>=c_med]; lo = subset[subset['cd276']<c_med]
            if len(hi)>=5 and len(lo)>=5:
                lr = logrank_test(hi['os_months'],lo['os_months'],hi['os_status'],lo['os_status'])
                mhi = KaplanMeierFitter().fit(hi['os_months'],hi['os_status']).median_survival_time_
                mlo = KaplanMeierFitter().fit(lo['os_months'],lo['os_status']).median_survival_time_
                print(f"  Within {label}: CD276-Hi={mhi:.1f}mo (n={len(hi)}) vs Lo={mlo:.1f}mo (n={len(lo)}), p={lr.p_value:.4f}")

# ============================================================
# CONCORDANCE INDEX SUMMARY
# ============================================================
print("\n"+"="*70); print("CONCORDANCE INDEX SUMMARY"); print("="*70)
print(f"\n  {'Model':45s} {'C-index':>8s} {'N':>6s}")
print(f"  {'-'*62}")
for k,v in mv_results.items():
    print(f"  {nice_labels.get(k,k):45s} {v['c']:>8.3f} {v['n']:>6d}")

# ============================================================
# COMPREHENSIVE SUMMARY
# ============================================================
print("\n"+"="*70); print("COMPREHENSIVE RESULTS SUMMARY"); print("="*70)
print("\n--- CD276 Survival ---")
for k,s in cd276_stats.items():
    print(f"  {k:20s}: n={s['n']}, Hi={s['median_hi']:.1f}mo, Lo={s['median_lo']:.1f}mo, \u0394={s['median_lo']-s['median_hi']:.1f}mo, p={s['logrank_p']:.4f}, HR={s['hr']:.2f}, C={s['c_index']:.3f}")
print("\n--- PD-L1 Survival ---")
for k,s in pdl1_stats.items():
    print(f"  {k:20s}: n={s['n']}, Hi={s['median_hi']:.1f}mo, Lo={s['median_lo']:.1f}mo, p={s['logrank_p']:.4f}, C={s['c_index']:.3f}")
print("\n--- Multivariate ---")
if '2013_uni' in mv_results and '2013_full' in mv_results:
    u=mv_results['2013_uni']; f=mv_results['2013_full']
    print(f"  Univariate:  HR={u['hr']:.3f} ({u['ci_lo']:.2f}-{u['ci_hi']:.2f}), p={u['p']:.4f}, C={u['c']:.3f}")
    print(f"  Full model:  HR={f['hr']:.3f} ({f['ci_lo']:.2f}-{f['ci_hi']:.2f}), p={f['p']:.4f}, C={f['c']:.3f}")
print("\n--- Gene Sets ---")
for gs in gsea_results:
    sig = '*' if gs['p']<0.05 else ''
    print(f"  {gs['set']:30s}: log2FC={gs['mean_l2fc']:+.3f}, p={gs['p']:.4f} {sig} | {gs['dir']}")

print("\n--- Files Generated ---")
for f in sorted(os.listdir(OUTDIR)):
    if f.endswith('.png'):
        print(f"  {f} ({os.path.getsize(os.path.join(OUTDIR,f))/1024:.0f} KB)")
print("\nDone!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.0/350.0 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 5.1 MB/s eta 0:00:00
LOADING ALL DATASETS

--- TCGA PanCancer ---
  Expr: data_mrna_seq_v2_rsem.txt shape=(20531, 161)
  Survival: 588 patients

--- TCGA Firehose ---
  Expr: data_mrna_seq_v2_rsem.txt shape=(20531, 167)
  Survival: 607 patients

--- TCGA 2013 ---
  Expr: data_mrna_seq_v2_rsem.txt shape=(20531, 153)
  Survival: 543 patients

--- CPTAC ---
  Expr: data_mrna_seq_fpkm.txt shape=(44963, 99)
  Survival: 97 patients

FIGURE 1: Prognostic Validation
  CD276: n=161, Hi=12.6 vs Lo=15.1, p=0.0365, HR=1.29, C=0.500
  CD276: n=97, Hi=13.2 vs Lo=21.2, p=0.1843, HR=1.12, C=0.566
  RNA-Protein: Spearman r=0.746 (p=8.4e-19), Pearson r=0.684, n=99
  -> Figure 1 saved

FIGURE 2: CD276 vs PD-L1
  CD276: n=161, Hi=12.6 vs Lo=15.1, p=0.0365, HR=1.29, C=0.500
  CD274: n=161, Hi=13.0 vs Lo=14.7, p=0.3931, HR=1.20, C=0.500
  -> Figure 2 saved
  PD-L1 prote

<lifelines.CoxPHFitter: fitted with 151 total observations, 52 right-censored observations>
             duration col = 'OS_time'
                event col = 'OS_event'
      baseline estimation = breslow
   number of observations = 151
number of events observed = 99
   partial log-likelihood = -386.29
         time fit was run = 2026-02-23 13:02:15 UTC

---
           coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                  
CD276_std  0.15      1.16      0.09           -0.04            0.33                0.96                1.39
Age        0.03      1.03      0.01            0.01            0.05                1.01                1.05
Sex_male  -0.25      0.78      0.21           -0.66            0.16                0.52                1.18

           cmp to     z      p  -log2(p)
covariate                               
CD276_std    0.00  1.54   0.12      3.01
Age          0.00  3.31 <0.005     10.08
Sex_male     0.00 -1.18   0.24      2.07
---
Concordance = 0.64
Partial AIC = 778.58
log-likelihood ratio test = 17.37 on 3 df
-log2(p) of ll-ratio test = 10.72


--- Full Multivariate (CD276 + Age + Sex + MGMT + IDH1) ---
  n=117 (after dropping missing)
  CD276 HR=1.338 (1.02-1.76), p=0.0384, C=0.697


<lifelines.CoxPHFitter: fitted with 117 total observations, 45 right-censored observations>
             duration col = 'OS_time'
                event col = 'OS_event'
      baseline estimation = breslow
   number of observations = 117
number of events observed = 72
   partial log-likelihood = -248.96
         time fit was run = 2026-02-23 13:02:16 UTC

---
             coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                    
CD276_std    0.29      1.34      0.14            0.02            0.57                1.02                1.76
Age          0.05      1.05      0.01            0.02            0.07                1.02                1.08
Sex_male    -0.03      0.97      0.28           -0.57            0.51                0.56                1.66
MGMT_unmeth  0.36      1.44      0.27           -0.17            0.90                0.84                2.45
IDH1_mut    -0.75      0.47      0.75           -2.22            0.72                0.11                2.06

             cmp to     z      p  -log2(p)
covariate                                 
CD276_std      0.00  2.07   0.04      4.70
Age            0.00  3.52 <0.005     11.19
Sex_male       0.00 -0.12   0.90      0.15
MGMT_unmeth    0.00  1.33   0.18      2.44
IDH1_mut       0.00 -1.00   0.32      1.66
---
Concordance = 0.70
Partial AIC = 507.91
log-likelihood ratio test = 26.75 on 5 df
-log2(p) of ll-ratio test = 13.93


--- Proportional Hazards Test ---
Proportional hazard assumption looks okay.
  PH assumption OK
  PanCancer univariate: HR=1.286, p=0.0016, C=0.572, n=161
  Firehose univariate: HR=1.282, p=0.0028, C=0.586, n=171
  CPTAC protein: HR=1.123, p=0.3328, C=0.566, n=96

GENE SET / PATHWAY ANALYSIS
  ABC_TRANSPORTERS              : 33 genes, mean log2FC=-0.141, p=0.1255 | DOWN in CD276-High
  DRUG_EFFLUX_CORE              :  7 genes, mean log2FC=-0.148, p=0.5781 | DOWN in CD276-High
  ADC_PAYLOAD_RESISTANCE        : 12 genes, mean log2FC=-0.058, p=0.9097 | DOWN in CD276-High
  STEM_CELL_MARKERS             : 13 genes, mean log2FC=+0.135, p=0.9460 | UP in CD276-High
  ANGIOGENESIS                  : 18 genes, mean log2FC=+0.405, p=0.0001 | UP in CD276-High
  MESENCHYMAL_GBM               : 16 genes, mean log2FC=+0.996, p=0.0001 | UP in CD276-High

SUPP FIGURE: Forest Plot
  TCGA 2013 (univariate)                  : HR=1.219 (1.02-1.46), p=0.0335, n=151
  TCGA 2013 (multivariate)              